To run this, press "*Runtime*" and press "*Run all*" on your A100 Google Colab Pro instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth your local device, follow [our guide](https://docs.unsloth.ai/get-started/install-and-update). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


### News


Introducing FP8 precision training for faster RL inference. [Read Blog](https://docs.unsloth.ai/new/fp8-reinforcement-learning).

Unsloth's [Docker image](https://hub.docker.com/r/unsloth/unsloth) is here! Start training with no setup & environment issues. [Read our Guide](https://docs.unsloth.ai/new/how-to-train-llms-with-unsloth-and-docker).

[gpt-oss RL](https://docs.unsloth.ai/new/gpt-oss-reinforcement-learning) is now supported with the fastest inference & lowest VRAM. Try our [new notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/gpt-oss-(20B)-GRPO.ipynb) which creates kernels!

Introducing [Vision](https://docs.unsloth.ai/new/vision-reinforcement-learning-vlm-rl) and [Standby](https://docs.unsloth.ai/basics/memory-efficient-rl) for RL! Train Qwen, Gemma etc. VLMs with GSPO - even faster with less VRAM.

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


### Installation

In [2]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Unsloth

`FastModel` supports loading nearly any model now! This includes Vision and Text models!

In [3]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-32B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


==((====))==  Unsloth 2025.12.5: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

We now add LoRA adapters so we only need to update a small amount of parameters!

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2025.12.5 patched 64 layers with 64 QKV layers, 64 O layers and 64 MLP layers.


In [5]:
from datasets import load_dataset

# Alpaca Formatı
alpaca_prompt = """Aşağıda bir görevi tanımlayan bir talimat ve daha fazla bağlam sağlayan bir girdi bulunmaktadır. İsteği uygun şekilde tamamlayan bir yanıt yazın.

### Instruction:
{}

### Input:
{}

### Output:
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

# Dosya adın farklıysa burayı düzelt
dataset = load_dataset("json", data_files="ed_dataset_upgraded.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)
print("✅ Veri seti hazır!")

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/3556 [00:00<?, ? examples/s]

✅ Veri seti hazır!


Let's see how the chat template did! Notice there is no `<bos>` token as the processor tokenizer will be adding one.

In [6]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 100,
        # max_steps satırını sildik, yerine epoch koyduk:
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/3556 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [7]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "### Instruction:\n",
    response_part = "### Output:\n",
)

Map (num_proc=16):   0%|          | 0/3556 [00:00<?, ? examples/s]

In [9]:

from google.colab import drive
drive.mount('/content/drive')
import os

print("🚀 Qwen 2.5 - 32B Eğitimi Başlıyor! Beklemede kal...")
trainer_stats = trainer.train()

# Drive'da oluşacak klasör adı
drive_model_path = "/content/drive/My Drive/EPDK_Qwen_32B_Uzman"

if not os.path.exists(drive_model_path):
    os.makedirs(drive_model_path)

print(f"💾 Model kaydediliyor: {drive_model_path}")
print("☕ Bu işlem 15-20 dk sürebilir, kapatma...")

model.save_pretrained_gguf(
    drive_model_path,
    tokenizer,
    quantization_method = "q4_k_m"
)
print("✅ İŞLEM BİTTİ! Drive'dan indirebilirsin.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 Qwen 2.5 - 32B Eğitimi Başlıyor! Beklemede kal...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,556 | Num Epochs = 3 | Total steps = 1,335
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 134,217,728 of 32,898,094,080 (0.41% trained)


Step,Training Loss
10,2.124600
20,1.993400
30,1.800000
40,1.721600
50,1.613900
60,1.664400
70,1.534500
80,1.506200
90,1.532900
100,1.432300


💾 Model kaydediliyor: /content/drive/My Drive/EPDK_Qwen_32B_Uzman
☕ Bu işlem 15-20 dk sürebilir, kapatma...
Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/709 [00:00<?, ?B/s]

...

Unsloth: Preparing safetensor model files: 100%|██████████| 14/14 [05:44<00:00, 24.62s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 14/14 [15:20<00:00, 65.75s/it] 


Unsloth: Merge process complete. Saved to `/content/drive/My Drive/EPDK_Qwen_32B_Uzman`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: All required system packages already installed!
Unsloth: Install llama.cpp and building - please wait 1 to 3 minutes
Unsloth: Cloning llama.cpp repository
Unsloth: Install GGUF and other packages
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...


RuntimeError: Unsloth: GGUF conversion failed: Unsloth: Failed to convert model to GGUF: Command 'python llama.cpp/unsloth_convert_hf_to_gguf.py --outfile qwen2.5-32b-instruct.BF16.gguf --outtype bf16 --split-max-size 50G /content/drive/My Drive/EPDK_Qwen_32B_Uzman' returned non-zero exit status 2.

In [18]:
!pip install sentence-transformers faiss-cpu

import json
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import TextStreamer

# 1. EPDK Kodları Listesi (Filtre İçin)
# Satırda bunlar yoksa, o satırı RAG'a almayacağız.
EPDK_KODLARI = [
    "EAG-", "FZG-", "İKG-", "OYS-", "KEY-",
    "VKY-", "TDY-", "EOG-", "ACG-", "İSG-",
    "TZY-", "ERY-", "KBS-"
]

dataset_path = "ed_dataset_upgraded.jsonl"
documents = []
doc_texts = []

print("📚 Mevzuat ayıklanıyor ve yükleniyor...")

filtered_count = 0
total_count = 0

try:
    with open(dataset_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: continue

            try:
                data = json.loads(line)
                total_count += 1

                # --- FİLTRELEME MEKANİZMASI ---
                # Metnin içinde EPDK kodu geçiyor mu?
                # Hem soruya (input) hem cevaba (output) bakıyoruz.
                icerik = data['input'] + " " + data['output']

                is_epdk_mevzuati = False
                for kod in EPDK_KODLARI:
                    if kod in icerik:
                        is_epdk_mevzuati = True
                        break

                # Eğer EPDK kodu yoksa (Genel sohbet ise) BU SATIRI ATLA
                if not is_epdk_mevzuati:
                    continue

                # ------------------------------

                # Sadece EPDK verilerini ekle
                full_text = f"Konu: {data['instruction']}\nDetay: {data['input']}\nÇözüm: {data['output']}"
                documents.append(data['output'])
                doc_texts.append(full_text)
                filtered_count += 1

            except json.JSONDecodeError:
                continue

    print(f"📊 İstatistik: Toplam {total_count} satırdan {filtered_count} adet SAF EPDK maddesi alındı.")
    print(f"🗑️ {total_count - filtered_count} adet genel sohbet verisi RAG'a dahil edilmedi.")

    # 2. Embedding ve FAISS Kurulumu
    print("🧠 Vektör veritabanı oluşturuluyor...")
    embedder = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = embedder.encode(doc_texts, convert_to_numpy=True)
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)
    print("✅ RAG Sistemi (Safkan EPDK) Hazır!")

except FileNotFoundError:
    print("❌ Dosya bulunamadı!")

# ---------------------------------------------------------
# SOHBET FONKSİYONU
# ---------------------------------------------------------
def ask_with_rag(soru):
    print(f"\n🔎 Mevzuat taranıyor: '{soru}'...")

    soru_vec = embedder.encode([soru], convert_to_numpy=True)

    # En alakalı 3 maddeyi getirsin
    k = 3
    D, I = index.search(soru_vec, k)

    context = ""
    print(f"✅ Bulunan Referanslar (Kopya):")
    for i in range(k):
        idx = I[0][i]
        if idx < len(documents):
            found_text = documents[idx]
            # Ekrana basarken kodun olduğu yeri gösterelim
            print(f"   📄 {found_text[:150]}...")
            context += f"{found_text}\n"

    # System Prompt
    system_instruction_rag = f"""Sen EPDK siber güvenlik denetçisisin.
Aşağıdaki 'MEVZUAT MADDELERİ'ni oku ve soruyu BUNA GÖRE cevapla.
Cevabında mutlaka ilgili madde numarasını (Örn: FZG-68) kullan.

MEVZUAT MADDELERİ:
{context}
"""

    alpaca_prompt = """### Instruction:
{}

### Input:
{}

### Output:
"""
    prompt = alpaca_prompt.format(system_instruction_rag, soru)

    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")
    text_streamer = TextStreamer(tokenizer, skip_prompt = True)

    print("\n🤖 UZMAN:", end=" ")
    _ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 1024, use_cache = True, temperature=0.1)

# TEST DÖNGÜSÜ
while True:
    soru = input("\n👤 SEN (Çıkmak için 'q'): ")
    if soru.lower() in ["q", "çık"]: break
    ask_with_rag(soru)

📚 Mevzuat ayıklanıyor ve yükleniyor...
📊 İstatistik: Toplam 3556 satırdan 1671 adet SAF EPDK maddesi alındı.
🗑️ 1885 adet genel sohbet verisi RAG'a dahil edilmedi.
🧠 Vektör veritabanı oluşturuluyor...
✅ RAG Sistemi (Safkan EPDK) Hazır!

👤 SEN (Çıkmak için 'q'): Sistem odasındaki yangın söndürme tüpleri pahalı geldi. Yerine standart kuru kimyevi tozlu (ABC tipi) tüp koysak veya su sistemi kursak mevzuata uygun olur mu?

🔎 Mevzuat taranıyor: 'Sistem odasındaki yangın söndürme tüpleri pahalı geldi. Yerine standart kuru kimyevi tozlu (ABC tipi) tüp koysak veya su sistemi kursak mevzuata uygun olur mu?'...
✅ Bulunan Referanslar (Kopya):
   📄 Kurum, 'Tekil Kaynak Bağımlılığı' (Single Point of Failure) oluşturan tedarikçileri (örn: Sadece tek bir firmanın bildiği özel bir yazılım) tespit etm...
   📄 Kurum, 'Tekil Kaynak Bağımlılığı' (Single Point of Failure) oluşturan tedarikçileri (örn: Sadece tek bir firmanın bildiği özel bir yazılım) tespit etm...
   📄 Kurum, 'Tekil Kaynak Bağımlılığı' (Si

Now let's print the masked out example - you should see only the answer is present:

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA A100-SXM4-40GB. Max memory = 39.557 GB.
19.074 GB of memory reserved.


Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100,000 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 56,758,272 of 27,489,164,912 (0.21% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,1.065200
2,1.393600
3,1.431700
4,1.121300
5,0.994200
6,1.409900
7,0.687700
8,1.071800
9,0.804700
10,0.691600


In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

291.7887 seconds used for training.
4.86 minutes used for training.
Peak reserved memory = 25.791 GB.
Peak reserved memory for training = 6.717 GB.
Peak reserved memory % of max memory = 65.2 %.
Peak reserved memory for training % of max memory = 16.981 %.


<a name="Inference"></a>
### Inference
Let's run the model via Unsloth native inference! According to the `Gemma-3` team, the recommended settings for inference are `temperature = 1.0, top_p = 0.95, top_k = 64`

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)
messages = [{
    "role": "user",
    "content": [{
        "type" : "text",
        "text" : "Continue the sequence: 1, 1, 2, 3, 5, 8,",
    }]
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = True,
    return_tensors = "pt",
    return_dict = True,
)
outputs = model.generate(
    **inputs.to("cuda"),
    max_new_tokens = 64, # Increase for longer outputs!
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
)
tokenizer.batch_decode(outputs)

['<bos><start_of_turn>user\nContinue the sequence: 1, 1, 2, 3, 5, 8,<end_of_turn>\n<start_of_turn>model\nThe sequence follows the rule of adding the previous two numbers. \n1+1 = 2\n1+2 = 3\n2+3 = 5\n3+5 = 8\n5+8 = 13\n8+13 = 21\nThe next two numbers in']

 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [ ]:
messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "Why is the sky blue?",}]
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = True,
    return_tensors = "pt",
    return_dict = True,
)

from transformers import TextStreamer
_ = model.generate(
    **inputs.to("cuda"),
    max_new_tokens = 64, # Increase for longer outputs!
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

The sky is blue because of a phenomenon called **Rayleigh scattering**. Here's an explanation:

1. **Sunlight consists of all colors.** When sunlight passes through the atmosphere, it interacts with air molecules (mainly nitrogen and oxygen).
2. **Shorter wavelengths scatter more.** Different colors of light


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("gemma-3")  # Local saving
tokenizer.save_pretrained("gemma-3")
# model.push_to_hub("HF_ACCOUNT/gemma-3", token = "...") # Online saving
# tokenizer.push_to_hub("HF_ACCOUNT/gemma-3", token = "...") # Online saving

['gemma-3/processor_config.json']

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastModel
    model, tokenizer = FastModel.from_pretrained(
        model_name = "gemma-3", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 2048,
        load_in_4bit = True,
    )

messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "What is Gemma-3?",}]
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = True,
    return_tensors = "pt",
    return_dict = True,
)

from transformers import TextStreamer
_ = model.generate(
    **inputs.to("cuda"),
    max_new_tokens = 64, # Increase for longer outputs!
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

Gemma is a family of open-weight models developed by Google DeepMind. It's designed to assist with a variety of tasks such as text generation, translation, and coding. There are two versions: Gemma 2B and Gemma 7B, which represent models with 2 billion and 7 billion parameters


### Saving to float16 for VLLM

We also support saving to `float16` directly for deployment! We save it in the folder `gemma-3-finetune`. Set `if False` to `if True` to let it run!

In [ ]:
if False: # Change to True to save finetune!
    model.save_pretrained_merged("gemma-3-finetune", tokenizer)

If you want to upload / push to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [ ]:
if False: # Change to True to upload finetune
    model.push_to_hub_merged(
        "HF_ACCOUNT/gemma-3-finetune", tokenizer,
        token = "hf_..."
    )

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now for all models! For now, you can convert easily to `Q8_0, F16 or BF16` precision. `Q4_K_M` for 4bit will come later!

In [ ]:
if False: # Change to True to save to GGUF
    model.save_pretrained_gguf(
        "gemma-3-finetune",
        tokenizer,
        quantization_method = "Q8_0", # For now only Q8_0, BF16, F16 supported
    )

Likewise, if you want to instead push to GGUF to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [ ]:
if False: # Change to True to upload GGUF
    model.push_to_hub_gguf(
        "HF_ACCOUNT/gemma-finetune-gguf",
        tokenizer,
        quantization_method = "Q8_0", # Only Q8_0, BF16, F16 supported
        token = "hf_...",
    )

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)
</div>
